In [1]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(Settings(anonymized_telemetry=False))

In [2]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

In [3]:
from vanna.chromadb import ChromaDB_VectorStore
from vanna.ollama import Ollama

class MyVanna(ChromaDB_VectorStore, Ollama):
    def __init__(self, config=None):
        ChromaDB_VectorStore.__init__(self, config=config)
        Ollama.__init__(self, config=config)

vn = MyVanna(config={
    "model": "llama3",
    "path": "/Users/vikram/Copilot/chroma_data"  # explicit, easy to find/delete
})

In [4]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["CHROMA_TELEMETRY"] = "False"

In [6]:
# ✅ CELL 3 — Connect to PostgreSQL
vn.connect_to_postgres(
    host="localhost",
    port=5432,
    dbname="postgres",
    user="vikram",
    password="vikram123"
)
print("Connected to PostgreSQL.")

Connected to PostgreSQL.


In [7]:
# ✅ CELL 4 — Train with DDL (run only once)
vn.train(ddl="""
CREATE TABLE jobs_data (
    job_id                  VARCHAR,
    job_title               VARCHAR,
    salary_usd              BIGINT,
    salary_currency         VARCHAR,
    experience_level        VARCHAR,
    employment_type         VARCHAR,
    company_location        VARCHAR,
    company_size            VARCHAR,
    employee_residence      VARCHAR,
    remote_ratio            BIGINT,
    required_skills         VARCHAR,
    education_required      VARCHAR,
    years_experience        BIGINT,
    industry                VARCHAR,
    posting_date            VARCHAR,
    application_deadline    VARCHAR,
    job_description_length  BIGINT,
    benefits_score          FLOAT,
    company_name            VARCHAR
);
""")
print("DDL training complete.")

Adding ddl: 
CREATE TABLE jobs_data (
    job_id                  VARCHAR,
    job_title               VARCHAR,
    salary_usd              BIGINT,
    salary_currency         VARCHAR,
    experience_level        VARCHAR,
    employment_type         VARCHAR,
    company_location        VARCHAR,
    company_size            VARCHAR,
    employee_residence      VARCHAR,
    remote_ratio            BIGINT,
    required_skills         VARCHAR,
    education_required      VARCHAR,
    years_experience        BIGINT,
    industry                VARCHAR,
    posting_date            VARCHAR,
    application_deadline    VARCHAR,
    job_description_length  BIGINT,
    benefits_score          FLOAT,
    company_name            VARCHAR
);



Add of existing embedding ID: a33fbe79-c95a-56ae-bbee-95fe0e985d74-ddl
Insert of existing embedding ID: a33fbe79-c95a-56ae-bbee-95fe0e985d74-ddl


DDL training complete.


In [8]:
# ✅ CELL 5 — Train with documentation (run only once)
vn.train(documentation="""
The jobs_data table has these exact column names:
- salary_usd is the salary in US dollars (not salary_in_usd)
- experience_level values are: SE (Senior), EN (Entry), MI (Mid), EX (Executive)
- employment_type values are: FT (Full-time), PT (Part-time), CT (Contract), FL (Freelance)
- company_size values are: S (Small), M (Medium), L (Large)
- remote_ratio is 0, 50, or 100 (percentage of remote work)
""")
print("Documentation training complete.")

Insert of existing embedding ID: 7bc7aeab-e8f7-5988-a092-f5e65bded326-doc
Add of existing embedding ID: 7bc7aeab-e8f7-5988-a092-f5e65bded326-doc


Adding documentation....
Documentation training complete.


In [9]:
# ✅ CELL 6 — Ask a natural language question
vn.ask("What are the top 5 highest paying job titles?")

Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1


SQL Prompt: [{'role': 'system', 'content': "You are a PostgreSQL expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \n\nCREATE TABLE jobs_data (\n    job_id                  VARCHAR,\n    job_title               VARCHAR,\n    salary_usd              BIGINT,\n    salary_currency         VARCHAR,\n    experience_level        VARCHAR,\n    employment_type         VARCHAR,\n    company_location        VARCHAR,\n    company_size            VARCHAR,\n    employee_residence      VARCHAR,\n    remote_ratio            BIGINT,\n    required_skills         VARCHAR,\n    education_required      VARCHAR,\n    years_experience        BIGINT,\n    industry                VARCHAR,\n    posting_date            VARCHAR,\n    application_deadline    VARCHAR,\n    job_description_length  BIGINT,\n    benefits_score          FLOAT,\n    company_name            VAR

Add of existing embedding ID: 87948730-c8ff-5c61-b3b3-d76147c5579b-sql
Insert of existing embedding ID: 87948730-c8ff-5c61-b3b3-d76147c5579b-sql


Info: Ollama Response:
model='llama3:latest' created_at='2026-05-26T15:29:26.149119Z' done=True done_reason='stop' total_duration=5488297875 load_duration=1665065417 prompt_eval_count=520 prompt_eval_duration=2230803417 eval_count=42 eval_duration=1580170750 message=Message(role='assistant', content='SELECT job_title, AVG(salary_usd) AS average_salary\nFROM jobs_data\nWHERE salary_usd IS NOT NULL\nGROUP BY job_title\nORDER BY average_salary DESC\nLIMIT 5;', thinking=None, images=None, tool_name=None, tool_calls=None) logprobs=None
LLM Response: SELECT job_title, AVG(salary_usd) AS average_salary
FROM jobs_data
WHERE salary_usd IS NOT NULL
GROUP BY job_title
ORDER BY average_salary DESC
LIMIT 5;
Info: Output from LLM: SELECT job_title, AVG(salary_usd) AS average_salary
FROM jobs_data
WHERE salary_usd IS NOT NULL
GROUP BY job_title
ORDER BY average_salary DESC
LIMIT 5; 
Extracted SQL: SELECT job_title, AVG(salary_usd) AS average_salary
FROM jobs_data
WHERE salary_usd IS NOT NULL
GROUP BY

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


('SELECT job_title, AVG(salary_usd) AS average_salary\nFROM jobs_data\nWHERE salary_usd IS NOT NULL\nGROUP BY job_title\nORDER BY average_salary DESC\nLIMIT 5',
                    job_title       average_salary
 0              AI Specialist  120570.758241758242
 1  Machine Learning Engineer  118827.919689119171
 2                 Head of AI  118542.968627450980
 3      AI Research Scientist  117897.925925925926
 4               AI Architect  117436.513618677043,
 Figure({
     'data': [{'hovertemplate': 'job_title=%{x}<br>average_salary=%{y}<extra></extra>',
               'legendgroup': '',
               'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
               'name': '',
               'orientation': 'v',
               'showlegend': False,
               'textposition': 'auto',
               'type': 'bar',
               'x': array(['AI Specialist', 'Machine Learning Engineer', 'Head of AI',
                           'AI Research Scientist', 'AI Architect'], dtyp

In [8]:
# ✅ CELL 7 — Launch the Flask web UI (optional)
# Opens a browser UI at http://localhost:8084
# Comment this out if you only want the Python API
from vanna.flask import VannaFlaskApp
app = VannaFlaskApp(vn)
app.run()

Your app is running at:
http://localhost:8084
 * Serving Flask app 'vanna.flask'
 * Debug mode: on


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 10 is greater than number of elements in index 1, updating n_results = 1
